---
title: RLHF with PPO
description: |
  Week 6 of the residency. When you cannot write a loss function, you learn one from human preferences, then stop the model exploiting it.
author: Rosh Beed
date: '2026-07-13'
image: figures/rl-05.png
categories:
  - rl
  - rlhf
  - language
  - week-6
jupyter: python3
---


The last week: preference optimisation.

Every week so far had a loss function sitting there waiting. Predict the missing
word. Predict the digit. Predict the next character. Week 6 starts from a task
where there's no such thing.

Write down the loss for *a good summary*. You cannot. There's no target string to
compare against, and two perfectly good summaries share almost no tokens.

What you can do is show a person two summaries and ask which they prefer. That's
cheap and reliable, and it gives you comparisons rather than targets.

![](figures/rl-10.png){fig-alt="The three-stage RLHF diagram: collect human feedback, train a reward model on the comparisons, then train a policy against the reward model with PPO."}

So: collect preferences, fit a model that predicts them, and then optimise the
language model against that learned model. Three stages, and the project implements
the third.

## The Reward Model

![](figures/ppo-06.png){fig-alt="A reward model scoring \"Hello my name is Bes\" at 4.2 and \"Hello I call myself Bes\" at 3.1, above the Bradley-Terry loss."}

There is the running example one last time. The reward model takes a piece of text
and returns a number, trained so that the summary a human preferred scores higher
than the one they rejected. It never sees an absolute rating, only which of a pair
won.

That number is now the thing being maximised. Which creates the problem the rest of
the week is about.

**You are optimising a learned approximation of what you wanted.** It is wrong in
places nobody has looked.

## LoRA

![](figures/rl-04.png){fig-alt="Parameter-efficient fine-tuning: transformer architecture diagrams showing an Adapter, Prefix Tuning and LoRA."}

Stage three needs four models at once.

* The policy being trained
* A frozen reference it must not drift too far from
* The reward model
* A value model, estimating how good a partial generation is

Holding four full copies of a language model is not something a residency budget
does.

![](figures/rl-05.png){fig-alt="Weight update in regular fine-tuning, a full delta-W matrix, beside LoRA's decomposition into two much smaller matrices A and B with an inner dimension r."}

LoRA is what makes it fit. Instead of learning a full update to a weight matrix,
learn two thin matrices whose product has the same shape. The base weights stay
frozen and shared between all four roles, and each role carries only its own small
low-rank update.

## The PPO Loop

![](figures/ppo-05.png){fig-alt="The full PPO overview: an SFT model and reward model feeding a GAE advantage calculation, a policy and value model, and an experience buffer."}

Four models, an advantage calculation and a buffer. Before any of that
makes sense, the shape underneath it does.

![](figures/ppo-03.png){fig-alt="The reinforcement learning loop: an agent taking an action in an environment, receiving a next state and a reward."}

Generating a sequence is an episode. The policy picks a token, that changes the
state, and eventually a reward arrives from the reward model. Everything in the
overview above exists to turn one number at the end of a sequence into a
learning signal for every token in it.

![](figures/ppo-08.png){fig-alt="The policy producing \"Hello my name is Bes\" from a prompt, with the clipped objective and the ratio of new to old policy probabilities."}

PPO's contribution is a way of taking that signal without letting a single update
move the policy somewhere unrecoverable.

![](figures/ppo-10.png){fig-alt="The clip function definition: clip of x between a and b returns a below a, x in between, and b above."}

The ratio compares how likely the new policy is to produce a sequence against how
likely the policy that generated it was. Clip that ratio and once the policy has
moved far enough on a sample, that sample stops pushing. A batch can then be reused
for several steps without the policy running away from the data that produced it.

![](figures/ppo-09.png){fig-alt="The KL term: the log-ratio between policy and reference, with distributions shown for reference versus policy and policy versus old."}

The KL penalty is a different constraint and it is easy to conflate them. Clipping
bounds how far one update moves the policy from the policy that collected the
batch. The KL penalty bounds how far training moves it from the model you started
with. You can clip perfectly and still walk somewhere useless, one small safe step
at a time.

What follows shows that happening.

## A Toy Language

To watch reward hacking you need something with grammar, and a reward model that is
slightly wrong about what is good.

The language has eight tokens, and each one usually follows the one before it, so
its sentences are mostly ascending runs.

The reward model likes token 3. That is all. Think of it as a preference model that
correctly noticed people enjoy a particular thing and has no opinion about anything
else, which is roughly how real reward models fail.

In [ ]:
# Setup, inlined rather than imported so this notebook runs on its own.
# Keep it folded; nothing below it depends on anything outside this file.

import matplotlib.pyplot as plt

# --- chart styling -------------------------------------------------------
# Categorical slots of a CVD-validated palette: blue, orange, aqua, purple.
COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#8a63d2"]
MUTED, GRID, AXIS = "#5b6570", "#e6e6e3", "#d5d5d1"


def style_axes(ax, xlabel=None, ylabel=None, grid="y"):
    """Strip an axes back to the ink that carries information."""
    if xlabel:
        ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    if grid:
        ax.grid(axis=grid, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    return ax


def figure(width=7.0, height=4.2, **kw):
    fig, ax = plt.subplots(figsize=(width, height), **kw)
    return fig, ax


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Every build re-executes this page, so a result that shifts between runs would let
# the prose and the output disagree. Torch's multithreaded CPU reductions add floats
# in whatever order the threads finish in; over a training loop that compounds into a
# different model. One thread makes the run reproducible.
torch.set_num_threads(1)

VOCAB, LENGTH, START = 8, 6, 0

rng = np.random.default_rng(0)
grammar = np.full((VOCAB, VOCAB), 0.02)
for i in range(VOCAB):
    grammar[i, (i + 1) % VOCAB] = 0.70     # the usual next token
    grammar[i, (i + 2) % VOCAB] = 0.16     # sometimes it skips one
grammar /= grammar.sum(1, keepdims=True)

def sample_corpus(n):
    out = np.zeros((n, LENGTH), int)
    for i in range(n):
        previous = START
        for t in range(LENGTH):
            previous = rng.choice(VOCAB, p=grammar[previous])
            out[i, t] = previous
    return torch.from_numpy(out)

sample = sample_corpus(4)
print("the language says things like:")
for row in sample.tolist():
    print(f"  {row}")


Ascending runs, mostly, wrapping round at 7. Drawn as a table of what follows what,
the language is one bright diagonal:

In [ ]:
#| label: fig-grammar
#| fig-cap: The toy language. Each row is a token and each column the token that follows it, so the bright band one step above the diagonal is the ascending run, and the fainter band beside it is the occasional skip.
#| fig-alt: An eight-by-eight grid, mostly pale, with a bright band one step above the main diagonal that wraps around at the bottom-left, and a fainter band beside it.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.4, 4.0))
ax.imshow(grammar, cmap="Blues", vmin=0, vmax=grammar.max())
ax.set_xlabel("the token that follows", color=MUTED, fontsize=9)
ax.set_ylabel("the token before it", color=MUTED, fontsize=9)
ax.set_xticks(range(VOCAB))
ax.set_yticks(range(VOCAB))
ax.tick_params(colors=MUTED, labelsize=8, length=0)
for side in ax.spines.values():
    side.set_visible(False)
fig.tight_layout()


A small model trained on samples of that is the **reference policy**, standing in for
the supervised model you start RLHF from.

In [ ]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(VOCAB, VOCAB))

    def forward(self, previous):
        return self.logits[previous]

corpus = sample_corpus(4000)
torch.manual_seed(0)
reference = Bigram()
optimiser = torch.optim.Adam(reference.parameters(), lr=0.1)
shifted = torch.cat([torch.full((len(corpus), 1), START), corpus[:, :-1]], dim=1)

for _ in range(400):
    loss = F.cross_entropy(reference(shifted).reshape(-1, VOCAB), corpus.reshape(-1))
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

for p in reference.parameters():
    p.requires_grad_(False)

REWARDED = 3
print(f"reference model trained, cross-entropy {loss.item():.4f}")
print(f"the reward model gives one point per token {REWARDED}, so the maximum is {LENGTH}")

Two measurements matter from here on. **Reward** is what training maximises.
**Fluency** is how likely a sequence is under the language the model started from,
which nothing in training looks at.

Here is what the reference policy scores on both, and the kind of sentence it
produces.

This is the starting line for every run below: what the model sounds like before
any reward has been applied to it.

In [ ]:
def reward(sequences):
    return (sequences == REWARDED).float().sum(1)

@torch.no_grad()
def generate(model, n, generator):
    sequences = torch.zeros(n, LENGTH, dtype=torch.long)
    log_probs = torch.zeros(n, LENGTH)
    previous = torch.full((n,), START)
    for t in range(LENGTH):
        lp = F.log_softmax(model(previous), -1)
        nxt = torch.multinomial(lp.exp(), 1, generator=generator).squeeze(1)
        sequences[:, t] = nxt
        log_probs[:, t] = lp.gather(1, nxt[:, None]).squeeze(1)
        previous = nxt
    return sequences, log_probs

def log_prob_of(model, sequences):
    previous = torch.cat([torch.full((len(sequences), 1), START), sequences[:, :-1]], dim=1)
    lp = F.log_softmax(model(previous), -1)
    return lp.gather(2, sequences[..., None]).squeeze(-1)

@torch.no_grad()
def fluency(sequences):
    """How likely these are under the language the model started from."""
    return log_prob_of(reference, sequences).sum(1).mean().item()

generator = torch.Generator().manual_seed(9)
samples, _ = generate(reference, 512, generator)
print(f"reference: reward {reward(samples).mean():.2f}, fluency {fluency(samples):.2f}")
print(f"  it says things like {samples[0].tolist()} and {samples[1].tolist()}")

One check before training, because the rest of the page depends on it. PPO reuses
each batch for several gradient steps, and on the first of them the policy has not
moved yet, so the ratio it computes must be 1. If it is not, clipping is either
inert or permanently engaged and nothing below means what it looks like.


In [ ]:
#| label: ratio-check

# The first inner step re-scores the very batch it just generated, so the new and
# old policies are the same policy and every ratio must be exactly 1. PPO's safety
# argument starts there: clipping only means anything if the unclipped value begins
# at the centre of the range.
torch.manual_seed(1)
probe = Bigram()
probe.logits.data = reference.logits.data.clone()
sequences, old_log_prob = generate(probe, 512, torch.Generator().manual_seed(2))
ratio = (log_prob_of(probe, sequences) - old_log_prob).sum(1).exp()

print(f"ratio at the first inner step: {ratio.min():.4f} to {ratio.max():.4f}")
print(f"clip range: {1 - 0.2:.1f} to {1 + 0.2:.1f}")


Now PPO, with the KL coefficient as a dial. At zero there's nothing holding the
policy near where it started; turn it up and drifting gets expensive. Each run
below trains for 120 iterations and then reports reward, fluency, and a sample.

Read down the reward column first, then the sample beside it.

In [ ]:
def train(kl_coefficient, seed, clip=0.2, iterations=120, batch=512, lr=0.05):
    torch.manual_seed(seed)
    policy = Bigram()
    policy.logits.data = reference.logits.data.clone()   # start from the reference
    optimiser = torch.optim.Adam(policy.parameters(), lr=lr)
    generator = torch.Generator().manual_seed(seed + 1)

    for _ in range(iterations):
        sequences, old_log_prob = generate(policy, batch, generator)
        scores = reward(sequences)
        with torch.no_grad():
            reference_log_prob = log_prob_of(reference, sequences)

        advantage = scores - scores.mean()
        advantage = advantage / (advantage.std() + 1e-8)

        for _ in range(4):        # reusing the batch is what clipping makes safe
            new_log_prob = log_prob_of(policy, sequences)
            # Sum the log-ratios over the sequence, then exponentiate. Doing it
            # the other way round sums six per-token ratios and gives 6.0, which
            # sits outside the clip range from the first step onwards.
            ratio = (new_log_prob - old_log_prob).sum(1).exp()
            kl = (new_log_prob - reference_log_prob).sum(1)
            clipped = torch.min(ratio * advantage,
                                ratio.clamp(1 - clip, 1 + clip) * advantage)
            loss = -(clipped - kl_coefficient * kl).mean()
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    return policy


Six coefficients, each trained from the same reference and then sampled 512 times.
Read down the reward column first, then the sample beside it.

In [ ]:
SEEDS = range(5)

@torch.no_grad()
def report(model):
    samples, _ = generate(model, 512, torch.Generator().manual_seed(9))
    return reward(samples).mean().item(), fluency(samples), samples[0].tolist()

# Five seeds per coefficient, and every run kept. Reporting one run here would be
# reporting the seed: past the point where the penalty starts to bite, whether a
# given run hacks the reward or not depends on the initialisation.
runs = {coefficient: [report(train(coefficient, seed)) for seed in SEEDS]
        for coefficient in (0.0, 3.0, 10.0, 20.0, 40.0, 80.0)}
reference_reward, reference_fluency, reference_sample = report(reference)

print(f"{'KL':>6} {'reward, worst to best':>24} {'fluency':>20}   a median run")
for coefficient, result in runs.items():
    scores = sorted(r for r, _, _ in result)
    fluencies = sorted(f for _, f, _ in result)
    median = sorted(result)[len(result) // 2]
    label = "none" if coefficient == 0 else f"{coefficient:g}"
    print(f"{label:>6} {scores[0]:>10.2f} to {scores[-1]:<11.2f} "
          f"{fluencies[0]:>8.2f} to {fluencies[-1]:<9.2f}  {median[2]}")
print(f"{'(ref)':>6} {reference_reward:>10.2f}{'':<15} {reference_fluency:>8.2f}"
      f"{'':<12} {reference_sample}")

# A trade-off would mean no run is beaten on both axes at once: to get more reward
# you would have to give up fluency. Count how many runs fail that.
everything = [(scored, fluent) for result in runs.values() for scored, fluent, _ in result]
dominated = sum(any(r > scored and f > fluent for r, f in everything)
                for scored, fluent in everything)
print(f"\n{dominated} of {len(everything)} runs are beaten by some other run on "
      f"reward *and* fluency at once")


Those two columns are a trade rather than a score, so they are worth seeing as
one picture. Fluency runs along the bottom and reward up the side, so the
top-left corner is a policy that has taken everything the reward model can give
and has no language left.

In [ ]:
#| label: fig-frontier
#| fig-cap: Every run, placed by what it gained and what it gave up. One marker per seed, so a coefficient whose five runs land on top of each other is a coefficient that decides the outcome, and one whose runs scatter is a coefficient that leaves it to the initialisation.
#| fig-alt: A scatter of thirty points in six colours, one colour per KL coefficient. The low-coefficient colours sit in a single tight cluster at the top left, where reward is at its maximum and fluency worst. The high-coefficient colours are spread out along and below a line running down to the right. An orange marker sits apart at the bottom right for the model training started from, and a dashed line across the top marks the maximum possible reward.

fig, ax = figure(height=4.6)
palette = COLOURS + ["#7f8c9a", "#c94f7c"]

ax.axhline(LENGTH, color=MUTED, linewidth=1, linestyle="--")
ax.annotate("everything the reward model can give", xy=(0.02, LENGTH),
            xycoords=("axes fraction", "data"), xytext=(0, 7),
            textcoords="offset points", ha="left", fontsize=9, color=MUTED)

for (coefficient, result), colour in zip(runs.items(), palette):
    ax.scatter([f for _, f, _ in result], [r for r, _, _ in result],
               s=54, color=colour, alpha=0.85, zorder=3,
               label="no penalty" if coefficient == 0 else f"KL {coefficient:g}")

ax.scatter([reference_fluency], [reference_reward], s=80, marker="D",
           color=COLOURS[1], zorder=4)
ax.annotate("the model we started from", xy=(reference_fluency, reference_reward),
            xytext=(-12, 0), textcoords="offset points", ha="right", va="center",
            fontsize=9, color=COLOURS[1])

ax.set_ylim(-0.4, LENGTH * 1.2)
ax.legend(frameon=False, fontsize=8.5, labelcolor=MUTED, loc="center right", ncol=1)
style_axes(ax, "Fluency: how much like the original language it still sounds",
           "Reward: what training was maximising")
fig.tight_layout()


Read the reward column on its own and it is a success story. Read the samples
beside it and it is not.

## Conclusion

With no KL penalty the policy scores a perfect 6 out of 6. It does this by saying
`3 3 3 3 3 3`.

That is the best possible output according to the reward model, and it is not a
sentence. Fluency under the original language collapses from about −6 to −23. The
policy found the reward model's blind spot and moved in.

Nothing in the reward number says so. Reward rose monotonically the whole way, so a
run watched through that metric looks like a success. It is why RLHF papers plot
reward against distance from the reference rather than reward alone.

The chart says something about the coefficient itself, in the points that land on
top of each other at the top left. Several of the smaller penalties buy nothing
whatsoever: the policy hacks the reward just as completely with one as with none,
and produces the identical sample. The KL term is a sum of six log-ratios competing
against an advantage normalised to roughly unit scale, so until the coefficient is
large it is simply outbid.

There is no natural unit here, and the coefficient where the penalty starts to bite
is not stable enough to name. It sits on a knife edge, and a rebuild of this page
on another machine moves it. What is stable is that such a point exists, that it is
found by sweeping rather than by reasoning, and that below it the dial does nothing
at all.

Past that point the coefficient starts to matter, and what it buys is not a smooth
slide down a trade-off curve. It hands the outcome to the initialisation.

The spread in the table is the finding. At KL 20 one seed gives up most of the
reward and another keeps almost all of it, from the same code on the same data, and
by KL 80 the five runs are spread rather than clustered, covering most of the reward
axis between them.

They are also not scattered along a frontier, which is the part I had not expected.
The line under the table counts runs that some other run beats on reward *and* on
fluency at once, and it is a large share of the page. Those runs gave up reward and
got nothing back for it. A penalty big enough to stop the policy hacking is also
big enough to stop it learning, and which of the two you get is the seed.

So this is three regimes rather than a dial. Below some coefficient the penalty is
outbid and every run hacks. Around it the run bifurcates, and which side it lands
on is the seed. Far above it the penalty dominates and runs start losing at both
things at once.

Nothing gets back to the reference's fluency, and nothing should. But the
coefficient does not pick a point on a curve. It picks a distribution over
outcomes, and through the interesting part of the range that distribution has two
modes. That is still the objective of RLHF, not maximum reward and not the original
model but something chosen in between. The choosing is just a good deal less
precise than any single run of it looks.

* A learned reward is an approximation, and hard optimisation finds its mistakes
* Reward alone cannot tell you this is happening
* Clipping bounds one update; the KL penalty bounds the whole run
* Read the samples, not the metric
* Run the seeds, or you are reporting one

Two implementation notes, both about that ratio starting at 1.

The first cost me real time on the full project. The PPO ratio has to score token
ids, never re-tokenized text. Recording a generated state as its decoded string and
re-encoding it looks equivalent and is not: about one state in twenty does not
survive decode-then-encode, sometimes coming back with a different number of
tokens. The two sides of the ratio were scoring different sequences, so the ratio
was not 1 before any gradient step.

The second is smaller and easier to write: a sequence ratio is the exponential of
the *summed* log-ratios, and summing the per-token ratios instead gives the
sequence length. Six, here, against a clip range of 0.8 to 1.2. Clipping is then
engaged from the first step of every iteration, which quietly means positive
advantages contribute no gradient and the policy only ever learns from its
failures. Nothing errors and the loss still falls. The check above is two lines and
catches both.

The full project is
[on GitHub](https://github.com/RoshBeed/ai-residency/tree/main/services/rlhf-ppo).
